# 06 Ablations - fold 1, seed 42

Inputs: hcsinhgoethe/ppg-cvae-prepared-fold01 and inputs/final/fold_01 via KRun --input.
Use selected beta=0.01; same architecture, optimizer and training budget as notebook 04.
no_hr_loss: retain HR conditioning, set lambda_hr=0.
no_condition: use zero condition and lambda_hr=0 (unconditional control).
Outputs: artifacts/ablations/{no_hr_loss,no_condition}/fold_01/seed_42.
These are validation diagnostics; test remains closed. Other folds are still pending.


In [1]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items


{'DATA_ROOT': '/kaggle/input/ppg-dalia-dataset', 'OUTPUT_ROOT': 'artifacts', 'device': 'cuda', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cu128'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Prepared dataset: https://www.kaggle.com/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01
import shutil
from zipfile import ZipFile
DATASET_SLUG = 'ppg-cvae-prepared-fold01'
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
fold_name = f'fold_{FOLD:02d}'
required_files = ('train.npz', 'val.npz', 'manifest.csv', 'normalization_stats.json', 'preprocessing_config.json')
destination = OUTPUT_ROOT/'prepared'/fold_name

# Discover the mounted dataset rather than assuming Kaggle's directory layout.
if ARTIFACT_INPUT_ROOT:
    dataset_roots = [Path(ARTIFACT_INPUT_ROOT)]
else:
    dataset_roots = sorted(p for p in input_dir.rglob(DATASET_SLUG) if p.is_dir()) if input_dir.exists() else []
if not dataset_roots:
    raise FileNotFoundError(f'Attach hcsinhgoethe/{DATASET_SLUG} with krun --dataset, or set PPGCVAE_INPUT_ROOT.')

prepared_sources = []
for root in dataset_roots:
    if not root.is_dir(): raise FileNotFoundError(root)
    for train_file in root.rglob('train.npz'):
        candidate = train_file.parent
        if candidate.name.startswith('fold_') and candidate.name != fold_name: continue
        if all((candidate/name).is_file() for name in required_files):
            prepared_sources.append(candidate)
if len(prepared_sources) > 1:
    raise ValueError(f'Ambiguous prepared artifact directories: {prepared_sources}; set PPGCVAE_INPUT_ROOT explicitly.')
if prepared_sources:
    prepared_source = prepared_sources[0]
    if prepared_source.resolve() != destination.resolve():
        shutil.copytree(prepared_source, destination, dirs_exist_ok=True)
    print('Imported prepared arrays from:', prepared_source)
else:
    # Also support a dataset whose uploaded ZIP has not been unpacked by Kaggle.
    archives = []
    for root in dataset_roots:
        for archive in root.rglob('*.zip'):
            with ZipFile(archive) as z:
                prefix = f'prepared/{fold_name}/'
                if all(prefix+name in z.namelist() for name in required_files):
                    archives.append(archive)
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one prepared {fold_name} directory or ZIP in {dataset_roots}; found archives: {archives}')
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as z:
        prefix = f'prepared/{fold_name}/'
        for name in (*required_files, 'test.npz', 'subject_summary.csv', 'hr_coverage.csv'):
            member = prefix+name
            if member in z.namelist():
                with z.open(member) as src, (destination/name).open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    print('Imported prepared arrays from ZIP:', archives[0])
assert all((destination/name).is_file() for name in required_files)
print('Prepared fold directory:', destination)

# Proxy is a differentiable estimate, not a ground-truth HR label.
# Preserve validation quality metadata in every tuning run for interpretation.
final_input = repo_root/'inputs'/'final'/fold_name
proxy_source = final_input/'proxy'
for name in ('hr_proxy.pt','detector_config.json','hr_proxy_validation_metrics.json','detector_validation_metrics.json'):
    if not (proxy_source/name).is_file():
        raise FileNotFoundError(f'Missing {proxy_source/name}; pass --input inputs/final/fold_01 to KRun.')
proxy_destination = OUTPUT_ROOT/'proxy'/fold_name
shutil.copytree(proxy_source,proxy_destination,dirs_exist_ok=True)
import json
proxy_quality = json.loads((proxy_destination/'hr_proxy_validation_metrics.json').read_text())
detector_quality = json.loads((proxy_destination/'detector_validation_metrics.json').read_text())
print('Proxy validation MAE bpm:',proxy_quality['best_val_mae_bpm'])
print('Detector real validation metrics:',detector_quality['selected'])

# Verify the transferred input package before using its selected configuration.
import hashlib
manifest = json.loads((final_input/'source_manifest.json').read_text(encoding='utf-8'))
assert manifest['fold'] == FOLD
for relative, expected in manifest['files_sha256'].items():
    actual = hashlib.sha256((final_input/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Input checksum mismatch: {relative}')
selected_source = final_input/'tuning'/'selected_config.json'
selected = json.loads(selected_source.read_text(encoding='utf-8'))
if selected['fold'] != FOLD:
    raise ValueError('Selected configuration belongs to a different fold')
tuning_destination = OUTPUT_ROOT/'tuning'/fold_name
shutil.copytree(final_input/'tuning', tuning_destination, dirs_exist_ok=True)
print('Selected configuration:', selected)
print('Ablation seed:', SEED)
print('Exploratory fold 1 run: current validation HR accuracy is low; test remains closed.')


FoldSubjects(fold_id=1, train_subjects=frozenset({'S8', 'S15', 'S9', 'S7', 'S12', 'S11', 'S14', 'S13', 'S10'}), val_subjects=frozenset({'S4', 'S5', 'S6'}), test_subjects=frozenset({'S3', 'S2', 'S1'}))


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01/prepared/fold_01
Prepared fold directory: artifacts/prepared/fold_01
Proxy validation MAE bpm: 11.888657971351767
Detector real validation metrics: {'prominence': 0.1, 'V': 0.9999155191349159, 'mae_hr': 9.10020403735541, 'A3': 0.4336402804764721}
Selected configuration: {'fold': 1, 'beta': 0.01, 'lambda_hr': 0.1, 'supported_targets': [60, 80, 120], 'validation_seed': 1042, 'num_val_samples': 200, 'selection_rule': 'V>=95% at supported targets, MAE, 0.1 bpm tie RMSE; fallback A3', 'selected_metrics': {'beta': 0.01, 'lambda_hr': 0.1, 'all_V_ge_95': True, 'mean_mae_hr': 19.693561122192943, 'mean_A3': 0.07833333333333332, 'val_rmse': 0.5547592043876648}}
Ablation seed: 42
Exploratory fold 1 run: current validation HR accuracy is low; test remains closed.


In [3]:
from ppg_cvae.workflow import train_selected
from ppg_cvae.utils.io import save_json
import pandas as pd

ABLATIONS = ('no_hr_loss', 'no_condition')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('This ablation job expects a GPU; run KRun with --gpu T4.')
if SEED != 42:
    raise ValueError('Mandatory ablations use seed 42.')
plan = dict(fold=FOLD, seed=SEED, beta=selected['beta'],
            reference_lambda_hr=selected['lambda_hr'], tuning_source_job=manifest['source_job'],
            ablations={'no_hr_loss': {'condition_mode': 'hr', 'lambda_hr': 0.0},
                       'no_condition': {'condition_mode': 'zero', 'lambda_hr': 0.0}},
            test_accessed=False)
save_json(plan, OUTPUT_ROOT/'ablations'/fold_name/'ablation_plan.json')
summaries = []
for ablation in ABLATIONS:
    print(f'Starting fold {FOLD}, seed {SEED}, ablation={ablation}', flush=True)
    run_dir, history = train_selected(OUTPUT_ROOT, FOLD, SEED, device=device, ablation=ablation)
    summary = pd.read_csv(run_dir/'prior_validation_summary.csv')
    summary['method'] = ablation
    summary['fold'] = FOLD
    summary['seed'] = SEED
    summaries.append(summary)
    print('Saved ablation:', run_dir, 'epochs:', len(history), flush=True)
    print(summary.to_string(index=False), flush=True)
pd.concat(summaries, ignore_index=True).to_csv(
    OUTPUT_ROOT/'ablations'/fold_name/'prior_validation_comparison.csv', index=False)
print('Completed both fold 1 ablations. Compare with cVAE seed 42 and the Gaussian baseline.')
print('No-condition also removes HR loss; compare it with no_hr_loss to isolate conditioning.')
print('Do not open final test evaluation until the protocol and all folds are ready.')


Starting fold 1, seed 42, ablation=no_hr_loss


Saved ablation: artifacts/ablations/no_hr_loss/fold_01/seed_42 epochs: 26


 target_hr    mae_hr  median_abs_error   V    A3  failed_windows  num_samples  mean_temporal_std  num_beats  rise_time_s_median  width50_s_median  rise_cycle_ratio_median  beat_extraction_success_median     method  fold  seed
        60 20.398239         21.702128 1.0 0.040               0          200           0.179287       1653              0.3125          0.421875                 0.418393                             1.0 no_hr_loss     1    42
        80  6.236484          5.333333 1.0 0.325               0          200           0.179500       1665              0.3125          0.414062                 0.417029                             1.0 no_hr_loss     1    42
       120 38.576660         37.419355 1.0 0.000               0          200           0.180477       1678              0.3125          0.406250                 0.420777                             1.0 no_hr_loss     1    42


Starting fold 1, seed 42, ablation=no_condition


Saved ablation: artifacts/ablations/no_condition/fold_01/seed_42 epochs: 25


 target_hr    mae_hr  median_abs_error   V    A3  failed_windows  num_samples  mean_temporal_std  num_beats  rise_time_s_median  width50_s_median  rise_cycle_ratio_median  beat_extraction_success_median       method  fold  seed
        60 19.752935         20.421053 1.0 0.020               0          200           0.171239       1659              0.3125          0.433594                 0.422089                             1.0 no_condition     1    42
        80  5.787432          5.333333 1.0 0.265               0          200           0.171239       1659              0.3125          0.433594                 0.422089                             1.0 no_condition     1    42
       120 40.247065         39.578947 1.0 0.000               0          200           0.171239       1659              0.3125          0.433594                 0.422089                             1.0 no_condition     1    42


Completed both fold 1 ablations. Compare with cVAE seed 42 and the Gaussian baseline.
No-condition also removes HR loss; compare it with no_hr_loss to isolate conditioning.
Do not open final test evaluation until the protocol and all folds are ready.
